In [ ]:
# Package imports

#import psi4
import psi4

# importing sympy for symbolic mantipulation
import sympy as sp

# import numpy for math and stuff
import numpy as np

# importing matplotlib module
from matplotlib import pyplot as plt

## Introduction to Computational Chemistry

Welcome to the dynamic field of computational chemistry!  In our class, we will be using the comptutational chemistry package psi4, which is a comprehensive electronic struture package.  Psi4 facilitates a range of computational experiments from basic to advanced, helping users methodically solve the Schrödinger equation with varying levels of approximation. You can learn more about Psi4, its features and capabilities at https://psicode.org/.

In most computational chemistry calculations, after you input the structure of the molecule you want to study, there are two critical choices you make:
1.  Level of Theory.  We recall that the reason you can not solve the Schrodinger Equation analytically is because of the electron correlation.  You must instead use a numerical algorithm to calculate the electron correlation.  The **level of theory** is the algorithm you choose to calculate the electron correlation.

The most foundational level of theory in electronic structure theory is Hartree-Fock theory.  In this method, each electron experiences only the *average* potentional of all the other electrons.  Other, more complete techniques include Density Functional Theory (DFT),  perturbation theory (MP2), Coupled Cluster (CC), and Full Configuration Interaction (FCI).

Each step in this hierarchy increases the precision of the results but also demands more computational resources, exemplifying the inherent trade-off between accuracy and algorithm runtime.

2.  Basis set.  The basis set is the set of functions you are going to use to build your wavefunction.  Very broadly speaking, the more functions you choose, the more accurate your calculation, but the longer your calculation takes.  In this lab we will compare four different basis sets: sto-3g, 6-31g, cc-pvdz, and cc-pvtz.

**The 'exact' answer (i.e. a computation that will match a perfect experimental value) is to use FCI in a very large basis set.**

Through this notebook, you'll learn to use psi4, apply different computational methods to the carbon monoxide molecule we studied previously, calculate molecular properties, and critically analyze the balance between computational expense and data fidelity.

## Example Psi4 code for calculating energies

**Important Note:** Psi4 creates intermediate files while it is running.  If your job fails for any reason, these files are not deleted.  If your job ever fails, all your jobs after that will probably fail too because you have these abandoned intermediate files.  If this happens, you should add the commands to clean up these files:
`psi4.core.clean()`, `psi4.core.clean_options()`, `
psi4.core.clean_variables()`.  If I am about to start a new section of calculations, I always clean up before I start.

In [ ]:
# Clean up before you start
psi4.core.clean()            # Clean temp files and scratch memory
psi4.core.clean_options()    # Reset all global options
psi4.core.clean_variables()

# Define the N2 molecule with 1.0 Å bond length

# Set your output file name
# The false option means you don't want the entire output to print
psi4.core.set_output_file('n2_calculations.out', False)

# Set up your molecule
n2 = psi4.geometry(f"""
N
N 1 1.0
""")

# Set the basis set
basis_set = 'cc-pVDZ'
psi4.set_options({'basis': basis_set})

# Perform HF calculation
hf_energy = psi4.energy('hf')
print(f'HF energy: {hf_energy:.12f} Eh')

# Perform MP2 calculation
mp2_energy = psi4.energy('mp2')
print(f'MP2 energy: {mp2_energy:.12f} Eh')

# Perform CCSD calculation
ccsd_energy = psi4.energy('ccsd')
print(f'CCSD energy: {ccsd_energy:.12f} Eh')

# Perform CCSD(T) calculation
ccsdt_energy = psi4.energy('ccsd(t)')
print(f'CCSD(T) energy: {ccsdt_energy:.12f} Eh')

**Timing Your Code**

As mentioned above there is an important tradeoff between computatoinal time and accuracy of calculation. There are several ways to time code, but a simple one is the `%timeit` "magic" command (for a single line), or `%%timeit` (for an entire cell) that can be used in any jupyter notebook.

To time any line of your code insert `%timeit -r 3 -n 3` at the begining of any line of code.  Typically I run the line of code and then duplicate it and add the `%timeit` command.  The `%timeit` command **only** prints your timing to the screen.  You can not save the output as a variable.  If you want to record your timings, you must write them down or record them in a seperate cell of your code.

In [ ]:
# Timing your code
# %timeit -r 3 -n 3

# Perform HF calculation
hf_energy = psi4.energy('hf')
%timeit -r 3 -n 3 hf_energy = psi4.energy('hf')
print(f'HF energy: {hf_energy:.12f} Eh')

# Perform MP2 calculation
mp2_energy = psi4.energy('mp2')
%timeit -r 3 -n 3 mp2_energy = psi4.energy('mp2')
print(f'MP2 energy: {mp2_energy:.12f} Eh')

# Perform CCSD calculation
ccsd_energy = psi4.energy('ccsd')
%timeit -r 3 -n 3 ccsd_energy = psi4.energy('ccsd')
print(f'CCSD energy: {ccsd_energy:.12f} Eh')

# Perform CCSD(T) calculation
ccsdt_energy = psi4.energy('ccsd(t)')
%timeit -r 3 -n 3 ccsdt_energy = psi4.energy('ccsd(t)')
print(f'CCSD(T) energy: {ccsdt_energy:.12f} Eh')


## Example code of Psi4 Geometry Optimization
In a geometry optimization, you have to perform many energy calculations.  You set a starting geometry and then calculate the energy of that geometry, then the geometry is changed slightly, and you calculate the energy of that new geometry.  If the new geometry is lower energy than the original geometry, you took a step in the right direction towards optimization.  This process continues until the energy is converged, that is, is doesn't change on subsequent steps.  Therefore, carefully choosing your computational parameters is even more important because you are going to do so many calculations.  In this example, we optimize the N$_2$ molecule and compare its energy to the energy of the molecule with a bond length of 1.0.  Note that, in psi4 by default, after you optimize the molecule, it will update the geometry to the optimized geometries and subsequent calculations will use that geometry.  This may be different in other electronic structure programs.

In [ ]:
# Perform geometry optimization with MP2
opt_energy = psi4.optimize('mp2')

#Saving and printing the optimized geometry
optimized_geometry = n2.save_string_xyz()
print(optimized_geometry)

# After the geometry optimization if you do any more energy calculations
# it will us the optimized geometry

# Example recalculate the mp2 energy and save it with a new name
mp2_energy_opt = psi4.energy('mp2')
print(f'MP2 energy at bond length 1.0 angstrom: {mp2_energy:.12f} Eh')
print(f'MP2 energy at optimized bond length: {mp2_energy_opt:.12f} Eh')


## Part 1: Comparing levels of theory

**Important Note.** Don't forget to clean up any old intermediate files before you start.
1.  Using the code example above as a guide, create a new CO molecule.  Make a reasonable guess for the bond length to the best of your ability.  Don't forget to set your output file to a new name also.
3.  Optimize the geometry of the molecule using computational method MP2/sto-3g.  Print the coordinates of your optimized molecule.
4.  Using the distance formula, calculate the optimized bond length.  (There are more and less clever ways to do this.  You can investigate clever ways if you want to.)
5.  Using the optimized geometry for CO, perform energy calculations for each of the following methods: HF, MP2, CCSD, CCSD(T), FCI.  Use the sto-3g basis set for all calculations. Time each energy calculation so you can compare them.
6.  Recall that, for any given basis set, the FCI energy is the most accurate energy calculation that can be done within that basis set.  Calculate the *signed* error for each energy method, Error = E of method - E for FCI.
7.  Make a scatter plot of the *signed* energy error with the method labeled on the x-axis.  Put the methods in order from the one that took the least time to the one that took the most time.   This is the plot you will turn in this week.
8.  What conclusions can you make about the quality of energy vs. the time calculations take?  Is doing the full CI "worth it"?  What method is the best time/cost tradeoff?


In [ ]:
# Clean up before you start
psi4.core.clean()            # Clean temp files and scratch memory
psi4.core.clean_options()    # Reset all global options
psi4.core.clean_variables()

# Your code goes here

## Part 2: Comparing Different Basis Sets

1. Clean up all your psi4 files before you start the new section.

2. Make a new CO molecule where the bond length is now your optimized bond length from before.

3. You are provided a list of six different basis sets.  The basis sets are listed in order of increasing size.  Using the optmized CO molecule, for each basis set, do an MP2 energy energy calculation.  Time each calculation.

4. Make a scatter plot of the calculated MP2 energies with the basis set labeled on the x-axis.  This is called a basis set convergence plot.  Taking into account the time for each basis set and the amount of improvement you got in the energy, what is the best basis set to use going forward?


In [ ]:
psi4.core.clean()            # Clean temp files and scratch memory
psi4.core.clean_options()    # Reset all global options
psi4.core.clean_variables()

psi4.core.set_output_file('basis_comparison.out', False)

# Set up your molecule here

In [ ]:
basis_list = ['sto-3g', '6-31g', 'cc-pvdz', 'cc-pvtz', 'cc-pvqz', 'cc-pv5z']

# Your code goes here

## Part 3: Getting the best equilibrium bond length for CO

With what you have learned so far, what would (in theory) be the best method/basis set combination you could use to optimize the CO bond length, balancing accuracy and computational cost?  

Use your chosen method and basis set to optimize the CO molecule and calculate the bond length of your optimized structure.  Compare it to the NIST experemenally accepted value of 112.8 pm.  Calculate the percent error.

You might even try a few different combinations and see if your additional calculation time improves your percent error significantly.

Never forget to clean up before you start!

In [ ]:
# Clean up
psi4.core.clean()            # Clean temp files and scratch memory
psi4.core.clean_options()    # Reset all global options
psi4.core.clean_variables()

# your code here!